#  AI Text Summarization using Fine-Tuned T5 Transformer

##  Project Overview

This notebook demonstrates the complete workflow for building an AI-powered text summarization system using the **T5-Small Transformer** model. The model is fine-tuned on the **SAMSum Dialogue Summarization Dataset** to generate concise and meaningful summaries from conversational text.

The notebook covers the complete pipeline, including:

- Dataset Loading
- Data Preprocessing
- Tokenization
- Model Fine-Tuning
- Model Saving
- Summary Generation
- Inference on Unseen Dialogues

**Frameworks Used :** PyTorch, Hugging Face Transformers, Pandas, NumPy

**Dataset :** SAMSum Dialogue Summarization Dataset

#  1. Import Libraries & Load Dataset : 

In this step, we import the essential Python libraries required for data manipulation and visualization. Next, the **SAMSum Dialogue Summarization      Dataset** is loaded into Pandas DataFrames for training and validation.

### Libraries Used
- **NumPy** – Numerical computations
- **Pandas** – Data loading and manipulation
- **Matplotlib** – Data visualization
- **Seaborn** – Statistical visualization
  

      Finally, the dimensions of both datasets are displayed to verify that the data has been loaded successfully.

In [1]:
# Load Dataset 
import numpy as np 
import pandas as pd 
import matplotlib.pyplot as plt
import seaborn as sns 

train_data = pd.read_csv(r"C:\Users\upadh\Downloads\samsum-train.csv")
val_data = pd.read_csv(r"C:\Users\upadh\Downloads\samsum-validation.csv")

print(train_data.shape)
print(val_data.shape)

(14732, 3)
(818, 3)


In [2]:
train_data.sample(10)

,id,dialogue,summary
9607,13863107,Gemma: Wait what? When did you leave?\nZac: li...,Zac left school two days ago after his mother'...
9508,13829251,"Abigail: hi darling, did you receive the Marti...",Abigail and Susan are going to the Martin's we...
4835,13681952,Aubrey: What about this dress?\r\nAubrey: <fil...,Aubrey and Anne are going to a clothing outlet...
9166,13814532,"Arthur: <file_other>\r\nOlivia: yeah, I've rea...",Arthur and Olivia read a negative article abou...
4371,13716062,"Lauren: sooo, do you guys know the gender yet?...",Abigail and Robert are going to have a baby gi...
541,13681949,Meggy: I am not sure if you've heard it but th...,Meggy and Brad's company is changing the offic...
9081,13730377,"Ramon: Stacey, what's happening tomorrow with ...",Maria is collecting exam papers tomorrow and c...
9225,13828470,Frank: you look amazin' today <3\r\nMacy: <3\r...,Frank likes Macy's red outfit.
13298,13819891,"Linda: you remember Mariusz, my English studen...",Linda found out that her ex student had passed...
7048,13612301,"Susan: Hi girlfriend,\r\nJacky: Hi,\r\nSusan:...",Jacky went out with Mike for a short while and...


# 2. Random Sampling of the Dataset : 

    To reduce computational cost and accelerate the fine-tuning process, a subset of the original dataset is selected.
    The dataset is randomly sampled using a fixed random seed to ensure reproducibility of the experiments.

- **Training Samples :** 4000
- **Validation Samples :** 500

In [3]:
# Random Sampling 
train_data = train_data.sample(n = 4000, random_state = 42).reset_index(drop = True)
val_data = val_data.sample(n = 500, random_state = 42).reset_index(drop = True)

print(train_data.shape)
print(val_data.shape)

(4000, 3)
(500, 3)


# 3. Text Preprocessing : 

    Before training the model, the dialogue and summary text are cleaned to improve data quality and consistency.

The preprocessing pipeline includes :

- Removing line breaks
- Removing extra whitespace
- Removing HTML tags
- Converting all text to lowercase
 

The cleaned text is then stored back into the dataset for further processing.

In [4]:
# Text Preprocessing 
import re 

def clean_data(text) : 
    text = re.sub(r"\r\n", " ", text) # Lines 
    text = re.sub(r"\s+", " ", text) # Spaces 
    text = re.sub(r"<.*?>", " ", text) # HTML Tags 
    text = text.strip().lower()
    return text 


train_data["dialogue"] = train_data["dialogue"].apply(clean_data)
train_data["summary"] = train_data["summary"].apply(clean_data)

val_data["dialogue"] = val_data["dialogue"].apply(clean_data)
val_data["summary"] = val_data["summary"].apply(clean_data)

In [5]:
train_data["dialogue"]

0       violet: hi! i came across this austin's articl...
1       pat: so does anyone know when the stream is go...
2       jane:   jane: whaddya think? shona: this ur ti...
3       adam: do u have a map of paris? tom: yes, why?...
4       frank: hi, how's the family? mike: great! sam'...
                              ...                        
3995    barry: hello buddy michael: hey barry: do you ...
3996    karen: hey lisa. larissa and me have recently ...
3997    miles: hey, guys, i'm so sorry, but i missed t...
3998    emma: did you finish the book i gave you? liam...
3999    jenna: dudes, were we supposed to read the who...
Name: dialogue, Length: 4000, dtype: object

# 4. Tokenization using T5 Tokenizer : 

The preprocessed dialogues and summaries are converted into numerical token IDs using the **T5 Tokenizer** from the Hugging Face Transformers library.

During tokenization:

- Dialogues are tokenized with a maximum length of **512 tokens**
- Summaries are tokenized with a maximum length of **150 tokens**
- Padding and truncation are applied to create fixed-length sequences

The tokenized summaries are assigned as labels for supervised fine-tuning.

In [6]:
# Tokenize 
from transformers import T5Tokenizer, Trainer, TrainingArguments, T5ForConditionalGeneration 

tokenizer = T5Tokenizer.from_pretrained("t5-small")

# Raw data => tokenized inputs for fine tuning 
def tokenize(data) : 
    inputs = tokenizer(data["dialogue"], padding = "max_length", max_length = 512, truncation = True)
    targets = tokenizer(data["summary"], padding = "max_length", max_length = 150, truncation = True)

    inputs["labels"] = targets["input_ids"]
    return inputs 

# 5. Prepare Training & Validation Datasets : 

    The tokenization function is applied to every dialogue-summary pair in both the training and validation datasets.

The processed data is converted into Python lists containing:

- Input IDs
- Attention Masks
- Target Labels

Finally, a sample tokenized record is displayed to verify the preprocessing pipeline.

In [7]:
train_dataset = train_data.apply(tokenize, axis = 1).tolist()
val_dataset = val_data.apply(tokenize, axis = 1).tolist()

train_dataset[0]

{'input_ids': [25208, 10, 7102, 55, 3, 23, 764, 640, 48, 403, 17, 77, 31, 7, 1108, 11, 3, 23, 816, 24, 25, 429, 253, 34, 1477, 25208, 10, 3, 7997, 15, 10, 7102, 55, 3, 10, 61, 2049, 6, 68, 3, 23, 31, 162, 641, 608, 34, 5, 3, 10, 61, 3, 7997, 15, 10, 68, 2049, 21, 1631, 81, 140, 3, 10, 61, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,

In [8]:
print(len(train_dataset[0]["input_ids"]))
print(len(val_dataset[0]["labels"]))

512
150


# 6. Load the Pre-trained T5 Model : 

The pre-trained **T5-Small Transformer** model is loaded from the Hugging Face model repository.

The notebook automatically detects the best available computing device:

- Apple MPS
- NVIDIA CUDA GPU
- CPU

The model is then moved to the detected device for efficient training and inference.

In [9]:
# Wroking with our model
import torch 

model = T5ForConditionalGeneration.from_pretrained("t5-small")

# Device configeration 
if torch.backends.mps.is_available() : 
    device = torch.device("mps")

elif torch.cuda.is_available() : 
    device = torch.device("cuda")

else : 
    device = torch.device("cpu")

print("Device : ", device)
model.to(device)

Loading weights:   0%|          | 0/131 [00:00<?, ?it/s]

Device :  cuda


T5ForConditionalGeneration(
  (shared): Embedding(32128, 512)
  (encoder): T5Stack(
    (embed_tokens): Embedding(32128, 512)
    (block): ModuleList(
      (0): T5Block(
        (layer): ModuleList(
          (0): T5LayerSelfAttention(
            (SelfAttention): T5Attention(
              (q): Linear(in_features=512, out_features=512, bias=False)
              (k): Linear(in_features=512, out_features=512, bias=False)
              (v): Linear(in_features=512, out_features=512, bias=False)
              (o): Linear(in_features=512, out_features=512, bias=False)
              (relative_attention_bias): Embedding(32, 8)
            )
            (layer_norm): T5LayerNorm()
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (1): T5LayerFF(
            (DenseReluDense): T5DenseActDense(
              (wi): Linear(in_features=512, out_features=2048, bias=False)
              (wo): Linear(in_features=2048, out_features=512, bias=False)
              (dropout): Drop

# 7. Fine-Tune the T5 Model : 

The training configuration is defined using Hugging Face's **TrainingArguments** and **Trainer** APIs.

Key training settings include:

- 6 Training Epochs
- Batch Size of 8
- Weight Decay Regularization
- Warm-up Steps
- Evaluation after every epoch
- Model checkpoint saving after every epoch

Finally, the model is fine-tuned on the SAMSum dataset.

In [10]:
# Training Arguments 
training_args = TrainingArguments(
    output_dir = "./results",
    num_train_epochs = 6,
    weight_decay = 0.01,
    per_device_train_batch_size = 8, 
    per_device_eval_batch_size = 8,
    eval_strategy = "epoch",
    save_strategy = "epoch",
    warmup_steps = 500
)

trainer = Trainer(
    model = model,
    args = training_args, 
    train_dataset = train_dataset, 
    eval_dataset = val_dataset
)

# Train the model 
trainer.train()

Epoch,Training Loss,Validation Loss
1,3.597911,0.380526
2,0.397470,0.359835
3,0.374047,0.354903
4,0.362774,0.350749
5,0.356223,0.349238
6,0.352467,0.349059


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=3000, training_loss=0.9068152720133463, metrics={'train_runtime': 1609.9338, 'train_samples_per_second': 14.907, 'train_steps_per_second': 1.863, 'total_flos': 3248203235328000.0, 'train_loss': 0.9068152720133463, 'epoch': 6.0})

# 8. Save the Fine-Tuned Model : 

    After training is complete, the fine-tuned model and tokenizer are saved locally.
    Saving the model allows it to be reused later for inference without repeating the training process.

    The saved model is then reloaded to verify that it has been stored successfully.

In [11]:
# Save and load our model 
model.save_pretrained("./saved_summary_model")
tokenizer.save_pretrained("./saved_summary_model")

model = T5ForConditionalGeneration.from_pretrained("./saved_summary_model")
tokenizer = T5Tokenizer.from_pretrained("./saved_summary_model")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/131 [00:00<?, ?it/s]

# 9. Build the Text Summarization Pipeline

A reusable summarization function is created to generate summaries for unseen dialogues.

The pipeline performs the following steps :

1. Clean the input text
2. Tokenize the dialogue
3. Generate the summary using the fine-tuned T5 model
4. Decode the generated token IDs into readable text

Beam Search decoding is used to improve the quality and coherence of the generated summaries.

In [12]:
# Test the core ;logic for summarization 
def summarize_dialogue(dialogue) : 
    dialogue = clean_data(dialogue)

    # Tokenize 
    inputs = tokenizer(
        dialogue, 
        padding = "max_length",
        max_length = 512, 
        truncation = True, 
        return_tensors = "pt"
    ).to(device)

    # Generate the summary => token ids 
    model.to(device)
    targets = model.generate(
        input_ids = inputs["input_ids"],
        attention_mask = inputs["attention_mask"],
        max_length = 150, 
        num_beams = 4,
        early_stopping = True
    )

    # Decode the output 
    summary = tokenizer.decode(targets[0], skip_special_tokens = True)
    return summary 

# 10. Model Inference : 

    In the final step, the trained model is evaluated on a new dialogue that was not part of the training dataset.

    The dialogue is passed through the complete summarization pipeline, and the generated summary is displayed to demonstrate the model's performance on unseen text.

In [13]:
# Testing 
test_dialogue = """ 
Reporter: In today's technology news, artificial intelligence continues to expand rapidly across industries, from healthcare to finance and education. Recent reports suggest that AI adoption has significantly increased over the past few years.

Reporter: Companies are investing heavily in machine learning systems to automate tasks, improve decision-making, and enhance customer experiences. However, this growth has also raised questions about job displacement and ethical concerns.

Expert: AI systems are becoming more capable due to advances in deep learning and access to large datasets. These models can now perform complex tasks such as language understanding, image recognition, and even code generation.

Expert: At the same time, there are valid concerns about bias in AI models, as they often reflect the data they are trained on. Ensuring fairness and transparency is becoming a key area of research.

Reporter: Governments and organizations are beginning to introduce regulations to guide the development and deployment of AI technologies. The goal is to balance innovation with accountability.

Expert: Another challenge is explainability. Many modern AI systems, especially deep neural networks, operate as “black boxes,” making it difficult to understand how decisions are made.

Reporter: Experts also highlight the importance of responsible AI development, including data privacy, security, and long-term societal impact.

Expert: Looking ahead, collaboration between researchers, policymakers, and industry leaders will be crucial to ensure that AI systems are developed and used in a safe and beneficial way.
"""



summary = summarize_dialogue(test_dialogue)
print("Summary : ", summary)

Summary :  ai adoption has significantly increased over the past few years. experts highlight the importance of responsible ai development, including data privacy, security, and long-term societal impact.
